# H3 云端 GPU 工作台 · Colab 通用 Worker

运行本 Notebook 后，Colab Pro+ 将自动接入 Cloudflare `gpu-workbench` 任务队列（与 Runpod / 本机 5090 共享同一套业务真源 `models_manifest.json` 与 `universal_worker.py`），直接复用 Google Drive `05_Colab/Models/minimax_h3` 里的权重（缺失项自动从 HF 补齐），每 5 秒上报实时进度并自动回传预览成片。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, getpass
os.environ['CONTROL_PLANE_URL'] = 'https://gpu-workbench.mamegoya6900.workers.dev'
os.environ['WORKER_PLATFORM'] = 'colab'
os.environ['MODELS_OVERRIDE_DIR'] = '/content/drive/MyDrive/05_Colab/Models/minimax_h3'
os.environ['WORKER_TOKEN'] = getpass.getpass('输入 cloud-console / gpu-workbench Token: ')

!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 >/dev/null
!if [ ! -d /content/ComfyUI ]; then git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI; fi
!pip install -q -r /content/ComfyUI/requirements.txt av huggingface_hub hf_transfer websocket-client requests
!cd /content/ComfyUI/custom_nodes && for repo in 'facok/comfyui-SelfLift' 'goohai/Goohaitools-comfyui' 'kijai/ComfyUI-KJNodes' 'Kosinkadink/ComfyUI-VideoHelperSuite'; do name=${repo#*/}; [ -d "$name" ] || git clone --depth 1 "https://github.com/${repo}.git" "$name"; [ -f "$name/requirements.txt" ] && pip install -q -r "$name/requirements.txt" || true; done
!curl -fsSL -H "Authorization: Bearer $WORKER_TOKEN" "$CONTROL_PLANE_URL/api/assets/runner/universal_worker.py" -o /content/universal_worker.py
!python3 -u /content/universal_worker.py --platform colab --comfy-dir /content/ComfyUI --work-dir /tmp/h3_work --models-dir "$MODELS_OVERRIDE_DIR"